# Preconditioned Crank-Nicolson (pCN) Tutorial

This tutorial demonstrates `ls_bayesian`'s function-space pCN sampler,
`PCNAlgorithm` (`ls_bayesian.mcmc.algorithms.pcn`), including its generalization to proposing from
an arbitrary Gaussian approximation of the target (Pinski, Simpson, Stuart, Weber, 2015) rather
than only from the prior itself (Cotter, Roberts, Stuart, White, 2013). To keep the tutorial
self-contained and fast to run, we use a linear-Gaussian toy inverse problem whose posterior is
known in closed form, so every sampler run can be checked against an exact reference.

## Mathematical Formulation

Function-space MCMC algorithms sample a target measure $\mu$ defined via its density relative to a
fixed reference measure $\mu_0$ (the problem's actual prior),

$$
\frac{d\mu}{d\mu_0} \propto \exp(-\Phi(u)).
$$

The classical pCN sampler proposes directly from $\mu_0$. Pinski et al. (2015) show that the
proposal can instead be drawn from any Gaussian measure $\nu = \mathcal N(\bar u_\nu, C_\nu)$
equivalent to $\mu_0$ -- e.g. a cheaper or better-fitting Gaussian approximation of $\mu$ itself --
at the cost of an extra term in the acceptance probability correcting for the mismatch between
$\nu$ and $\mu_0$. Given the current state $u$ and step width $\delta \in (0,1)$, the proposal is

$$
v = \bar u_\nu + \sqrt{1-\delta^2}\,(u - \bar u_\nu) + \delta\, w, \qquad w \sim \mathcal
N(0, C_\nu),
$$

with acceptance probability

$$
\alpha(u,v) = 1 \wedge \exp\big(\Phi_\nu(u) - \Phi_\nu(v)\big), \qquad \Phi_\nu(u) = \Phi(u) -
\Phi_\nu^\text{approx}(u),
$$

where $\Phi_\nu^\text{approx}$ is $\nu$'s own potential relative to $\mu_0$,
$\frac{d\nu}{d\mu_0} \propto \exp(-\Phi_\nu^\text{approx}(u))$. Choosing $\nu = \mu_0$ itself
(identically zero $\Phi_\nu^\text{approx}$) recovers classical pCN exactly. This tutorial first
runs `PCNAlgorithm` in that classical mode, then again with an informed $\nu$ to show how a good
approximation lets pCN take much larger steps without losing acceptance.

References:

- Cotter, Roberts, Stuart, White (2013). *MCMC Methods for Functions: Modifying Old Algorithms to
  Make Them Faster.* Statistical Science 28(3).
- Pinski, Simpson, Stuart, Weber (2015). *Algorithms for Kullback-Leibler Approximation of
  Probability Measures in Infinite Dimensions.* SIAM Journal on Scientific Computing 37(6),
  A2733-A2757.

## Imports and Configuration

We import NumPy for the linear algebra, Matplotlib for the acceptance-rate plots,
`typing.override` for the measure/target implementations, and the `algorithms.pcn`, `measures`,
`sampler`, and `output` modules of `ls_bayesian`'s `mcmc` subpackage. All randomness is seeded for
reproducibility.

In [ ]:
from typing import override

import matplotlib.pyplot as plt
import numpy as np

from ls_bayesian.mcmc import output as mcmc_output
from ls_bayesian.mcmc.algorithms.pcn import PCNAlgorithm
from ls_bayesian.mcmc.measures import GaussianMeasure, TargetMeasure
from ls_bayesian.mcmc.model import MCMCModel
from ls_bayesian.mcmc.sampler import Sampler, SamplerSettings
from ls_bayesian.mcmc.storage import NumpyStorage

rng = np.random.default_rng(0)
STATE_DIM = 4

## A Linear-Gaussian Test Problem

Consider a parameter $u \in \mathbb R^n$ with quadratic potential
$\Phi(u) = \frac{1}{2}(u-a)^T H (u-a)$ (a linearized, Gauss-Newton-style misfit around minimizer
$a$ with Hessian $H$) and a centered Gaussian reference $\mu_0 = \mathcal N(0, C_0)$. The resulting
target $\mu$ is then itself Gaussian, with precision $H + C_0^{-1}$ and mean
$(H+C_0^{-1})^{-1} H a$ -- giving us an exact reference to check every sampler run against, exactly
as `tests/mcmc/helpers.py` does for this package's own stationarity tests.

In [ ]:
def random_spd_matrix(rng: np.random.Generator, dim: int) -> np.ndarray:
    """Return a random symmetric positive-definite matrix of shape (dim, dim)."""
    factor = rng.random((dim, dim))
    return factor @ factor.T + dim * np.eye(dim)


hessian = random_spd_matrix(rng, STATE_DIM)
prior_covariance = random_spd_matrix(rng, STATE_DIM)
minimizer = rng.standard_normal(STATE_DIM)

prior_precision = np.linalg.inv(prior_covariance)
posterior_precision = hessian + prior_precision
posterior_covariance = np.linalg.inv(posterior_precision)
posterior_mean = posterior_covariance @ (hessian @ minimizer)
posterior_standard_deviation = np.sqrt(np.diag(posterior_covariance))

print("Analytic posterior mean:", posterior_mean)
print("Analytic posterior marginal std :", posterior_standard_deviation)

## Implementing the Target

`PCNAlgorithm` needs $\Phi$ through the `TargetMeasure` interface -- pCN is derivative-free, so unlike the `pmala` tutorial's target, this one has no `evaluate_gradient` at all.

In [ ]:
class QuadraticTargetMeasure(TargetMeasure):
    """Quadratic potential Phi(u) = 1/2 (u-a)^T H (u-a)."""

    def __init__(self, matrix: np.ndarray, minimizer: np.ndarray) -> None:
        self.matrix = matrix
        self.minimizer = minimizer

    @override
    def evaluate_potential(self, state: np.ndarray) -> float:
        difference = state - self.minimizer
        return float(0.5 * difference @ self.matrix @ difference)


target = QuadraticTargetMeasure(hessian, minimizer)

## Classical pCN: Proposing From the Prior

For classical pCN, `approximation` is left `None`, so `PCNAlgorithm` proposes from `reference`
($\mu_0$) itself and the correction relative to it is identically zero; the next section instead
passes an `approximation` representing an informed $\nu \neq \mu_0$.

In [ ]:
class DenseGaussianMeasure(GaussianMeasure):
    """Gaussian measure nu = N(mean, covariance): PCNAlgorithm derives the Pinski et al. (2015)
    correction relative to mu_0 automatically from this class's mean/precision, so this class
    itself carries no correction-specific logic -- the same class serves as both mu_0 (`prior_measure`
    below) and any nu proposed from instead (`informed_measure` below)."""

    def __init__(self, mean_vector: np.ndarray, covariance_matrix: np.ndarray) -> None:
        self.mean_vector = mean_vector
        self.covariance_matrix = covariance_matrix
        self.precision_matrix = np.linalg.inv(covariance_matrix)
        self.covariance_factor = np.linalg.cholesky(covariance_matrix)

    @property
    @override
    def mean(self) -> np.ndarray:
        return self.mean_vector

    @property
    @override
    def random_vector_size(self) -> int:
        return self.covariance_factor.shape[1]

    @override
    def apply_covariance_factorization(self, random_vector: np.ndarray) -> np.ndarray:
        return self.covariance_factor @ random_vector

    @override
    def apply_covariance_operator(self, vector: np.ndarray) -> np.ndarray:
        return self.covariance_matrix @ vector

    @override
    def apply_precision_operator(self, vector: np.ndarray) -> np.ndarray:
        return self.precision_matrix @ vector


prior_measure = DenseGaussianMeasure(np.zeros(STATE_DIM), prior_covariance)

## Running the Sampler

`Sampler` drives `PCNAlgorithm.compute_step` for a fixed number of samples, dispatching each new
state to `NumpyStorage` and to an `MCMCOutput` tracking the running-mean acceptance rate
(`mcmc_output.build` derives its column label/format automatically). The step width $\delta$
trades off acceptance against exploration: small $\delta$ stays close to $\mu_0$'s own step
distribution (high acceptance, slow mixing), while larger $\delta$ explores faster until $\nu =
\mu_0$'s mismatch with the (comparatively concentrated) posterior collapses the acceptance rate --
we quantify that trade-off in a step-width sweep below.

In [ ]:
BURN_IN = 2000
NUM_SAMPLES = 20_000


def run_pcn_chain(
    approximation: GaussianMeasure | None, step_width: float, seed: int
) -> tuple[np.ndarray, float]:
    """Run PCNAlgorithm for BURN_IN + NUM_SAMPLES steps and return the post-burn-in samples and
    the final running-mean acceptance rate."""
    model = MCMCModel(target=target, reference=prior_measure, approximation=approximation)
    algorithm = PCNAlgorithm(model, step_width)
    acceptance_output = mcmc_output.build(
        mcmc_output.AcceptanceQoI(), mcmc_output.RunningMeanStatistic()
    )
    storage = NumpyStorage()
    sampler = Sampler(algorithm, storage=storage, outputs=[acceptance_output])
    settings = SamplerSettings(
        num_samples=BURN_IN + NUM_SAMPLES, log_interval=BURN_IN + NUM_SAMPLES
    )
    sampler.run(posterior_mean.copy(), settings, seed=seed)
    return storage.values[BURN_IN:], acceptance_output.value


classical_samples, classical_acceptance_rate = run_pcn_chain(None, step_width=0.1, seed=1)
print(f"Classical pCN acceptance rate: {classical_acceptance_rate:.3f}")

## Verifying the Recovered Posterior Moments

The empirical mean and standard deviation of the post-burn-in samples should match the analytic
posterior moments computed above, up to Monte Carlo error.

In [ ]:
def report_moment_recovery(samples: np.ndarray, label: str) -> None:
    """Print the sample mean/std next to the analytic reference, normalized by the analytic
    standard deviation so the comparison is scale-free across coordinates."""
    sample_mean = samples.mean(axis=0)
    sample_std = samples.std(axis=0, ddof=1)
    normalized_mean_error = (sample_mean - posterior_mean) / posterior_standard_deviation
    print(f"{label}:")
    print("  normalized mean error:", normalized_mean_error)
    print("  sample std / analytic std:", sample_std / posterior_standard_deviation)
    assert np.all(np.abs(normalized_mean_error) < 0.5)
    assert np.all(np.abs(sample_std / posterior_standard_deviation - 1.0) < 0.3)


report_moment_recovery(classical_samples, "Classical pCN")

## Step-Width Sweep

Sweeping $\delta$ over $(0,1)$ makes the acceptance/exploration trade-off concrete: acceptance
falls off sharply once the proposal's scale starts to compete with the target's own, comparatively
small, posterior covariance.

In [ ]:
def sweep_acceptance_rate(
    approximation: GaussianMeasure | None, step_widths: np.ndarray
) -> np.ndarray:
    """Return the final running-mean acceptance rate for each step width in `step_widths`, from a
    short chain (fine for an acceptance-rate estimate, unlike the moment-recovery runs above)."""
    acceptance_rates = np.empty_like(step_widths)
    for index, step_width in enumerate(step_widths):
        _, acceptance_rates[index] = run_pcn_chain(approximation, float(step_width), seed=1)
    return acceptance_rates


sweep_step_widths = np.array([0.02, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 0.95])
classical_acceptance_sweep = sweep_acceptance_rate(None, sweep_step_widths)
print(classical_acceptance_sweep)

## Generalized pCN: Proposing From an Informed Approximation

Now we replace $\nu = \mu_0$ with an informed Gaussian approximation of the target itself: $\nu =
\mathcal N(\bar u_\text{post}, C_\text{post})$, the *exact* analytic posterior computed above (in
practice this would instead be, e.g., a Laplace approximation computed once at the MAP estimate).
`PCNAlgorithm` derives the correction potential $\Phi_\nu(u)$ for any such $\nu$ automatically from
`mu_0`'s and `nu`'s own potentials (Pinski et al., 2015), so building the informed proposal is a
one-line change from the classical one.

In [ ]:
informed_measure = DenseGaussianMeasure(posterior_mean, posterior_covariance)

informed_samples, informed_acceptance_rate = run_pcn_chain(
    informed_measure, step_width=0.9, seed=1
)
print(f"Generalized pCN acceptance rate (step_width=0.9): {informed_acceptance_rate:.3f}")
report_moment_recovery(informed_samples, "Generalized pCN")

Because $\nu$ coincides with the target itself here, $\Phi_\nu(u) = \Phi(u) -
\Phi_\nu^\text{approx}(u)$ is constant in $u$ (up to the additive normalizer dropped above), so the
acceptance probability is exactly $1$ for *every* state and step width -- an idealized best case,
but one that demonstrates the point of the generalization: a good-enough Gaussian approximation
lets pCN draw large, well-mixed steps at essentially no cost in acceptance, something classical pCN
cannot do once its proposal's scale departs from the target's own.

In [ ]:
informed_acceptance_sweep = sweep_acceptance_rate(informed_measure, sweep_step_widths)

fig, ax = plt.subplots()
ax.plot(sweep_step_widths, classical_acceptance_sweep, marker="o", label=r"classical ($\nu=\mu_0$)")
ax.plot(
    sweep_step_widths,
    informed_acceptance_sweep,
    marker="o",
    label=r"informed ($\nu=$ posterior)",
)
ax.set_xlabel(r"step width $\delta$")
ax.set_ylabel("acceptance rate")
ax.set_ylim(-0.05, 1.05)
ax.set_title("pCN acceptance rate vs. step width")
ax.legend()
fig.tight_layout()

## Summary

- Classical pCN (`approximation=None`, i.e. $\nu = \mu_0$) trades acceptance for exploration
  through a single step width $\delta \in (0,1)$; both runs above recovered the analytic posterior
  mean and standard deviation to within Monte Carlo error.
- The generalized pCN sampler (Pinski et al., 2015) reuses the exact same `PCNAlgorithm` and
  `Sampler`/`NumpyStorage`/`MCMCOutput` machinery, only swapping in an `MCMCModel.approximation`
  whose mismatch relative to `mu_0` is corrected for automatically -- here, an idealized informed
  approximation that keeps acceptance at $1$ across the entire swept range of step widths, in
  contrast to classical pCN's sharp decay.